# Namma Space — Cloud 3D Gaussian Splatting Reconstruction Pipeline
**Techfest IIT Bombay 2026–27 | Round 1**

This notebook trains an authentic 3D Gaussian Splatting digital twin from your smartphone walkthrough video using an NVIDIA T4 GPU on Google Colab.

---
### Pre-Flight Checklist:
1. **Runtime Type**: Go to **Runtime → Change runtime type** and select **T4 GPU**.
2. **Recorded Video**: A 45–75 second MP4/MOV recorded following `docs/CAPTURE_SOP.md` (1080p landscape, locked AE/AF, dual-loop).

### Step 1: Verify NVIDIA GPU & CUDA

In [ ]:
!nvidia-smi

### Step 2: Clone Repositories & Install Dependencies (PyColmap + Inria 3DGS CUDA Rasterizer)

In [ ]:
# 1. Clone Roomsplat pipeline orchestrator
!git clone https://github.com/Ahmedn1/roomsplat.git
%cd /content/roomsplat
!pip install -e . pycolmap typer rich

# 2. Clone official Inria 3D Gaussian Splatting engine
%cd /content
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git
%cd /content/gaussian-splatting
!pip install submodules/diff-gaussian-rasterization
!pip install submodules/simple-knn

# 3. Link Gaussian Splatting directory to Roomsplat
!roomsplat config --gs-dir /content/gaussian-splatting

print("\nSUCCESS: Environment & CUDA rasterizers compiled successfully!")

### Step 3: Upload Your Smartphone Walkthrough Video

In [ ]:
from google.colab import files
import os

%cd /content
print("Upload your smartphone video (e.g. my_room.mp4):")
uploaded = files.upload()
video_filename = list(uploaded.keys())[0]
print(f"\nUploaded video: {video_filename} ({os.path.getsize(video_filename) / (1024*1024):.2f} MB)")

### Step 4: Run End-to-End Reconstruction (Extraction → SfM → 3DGS → Splat Export)

In [ ]:
%cd /content
# Recommended settings for 45-75s room video on T4 GPU (approx. 20-30 min training)
!roomsplat run "{video_filename}" \
    --name "Namma Space Authentic Room" \
    --fps 1.5 \
    --max-dim 1600 \
    --iterations 30000 \
    --low-vram \
    --no-view

print("\nSUCCESS: 3D Gaussian Splatting reconstruction finished!")

### Step 5: Download `scene.splat` and `cameras.json` for Namma Space Viewer

In [ ]:
import glob
import shutil
from google.colab import files

# Find output directory
splat_dirs = glob.glob("/content/*_splat")
if splat_dirs:
    out_dir = splat_dirs[0]
    print(f"Found reconstruction output at: {out_dir}")
    
    # Bundle the two files needed for the Namma Space web viewer
    splat_file = f"{out_dir}/scene.splat"
    cameras_file = f"{out_dir}/cameras.json"
    
    shutil.copy(splat_file, "/content/scene.splat")
    shutil.copy(cameras_file, "/content/cameras.json")
    
    # Zip together for easy download
    !zip -j /content/namma_space_assets.zip /content/scene.splat /content/cameras.json
    
    print("\nDownloading namma_space_assets.zip to your computer...")
    files.download("/content/namma_space_assets.zip")
else:
    print("Error: Output splat directory not found!")